[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/31_transformers/01_attention_from_scratch.ipynb)

# Bài 01 — Attention from scratch

**🎯 Mục tiêu**
- Hiểu attention như một phép **tra cứu mềm** (soft dictionary lookup): query, key, value.
- Cài **scaled dot-product attention** bằng NumPy, kiểm tra khớp PyTorch; hiểu **vì sao chia $\sqrt{d_k}$** qua thí nghiệm.
- **Mask**: padding mask, causal mask.
- **Self-attention**: tính hoán vị-đẳng biến (permutation equivariance) → vì sao cần mã hoá vị trí.
- Thấy attention giải được bài toán tra cứu mà mô hình "gộp trung bình" lớn hơn 20 lần không giải được; chi phí $O(n^2)$.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/31_transformers

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from ml_course.visualization import set_style

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.deterministic = True
print("thiết bị:", device)

## 1. Attention là tra cứu mềm

Một từ điển Python tra cứu **cứng**: so khoá truy vấn với từng khoá, trả về **đúng một** giá trị. Attention làm phiên bản **mềm, khả vi**:

1. So **query** $\mathbf q$ với mọi **key** $\mathbf k_j$ bằng tích vô hướng → điểm $e_j = \mathbf q^\top\mathbf k_j$.
2. Chuẩn hoá bằng softmax → trọng số $\alpha_j \ge 0$, $\sum_j\alpha_j = 1$.
3. Trả về **trung bình có trọng số** của các **value**: $\sum_j\alpha_j\mathbf v_j$.

Vì mọi bước đều khả vi, mạng có thể **học** nên hỏi gì (query), đánh dấu gì (key) và trả về gì (value). Ở chương 30, decoder dùng attention để "nhìn lại" các trạng thái encoder; Transformer dùng nó làm phép toán **chính**.

In [ ]:
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)                             # trừ max: ổn định số
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)


angles = np.deg2rad([0, 60, 120, 180])
keys = np.c_[np.cos(angles), np.sin(angles)]                            # 4 khoá: vector đơn vị
values = np.array([10.0, 20.0, 30.0, 40.0])
query = np.array([np.cos(np.deg2rad(70)), np.sin(np.deg2rad(70))])      # gần khoá thứ 2 (60°)

for sharpness in [1, 5, 50]:
    w = softmax(sharpness * keys @ query)
    print(f"độ sắc {sharpness:2d}: trọng số = {np.round(w, 3)} → giá trị trả về = {w @ values:.2f}")
print("tra cứu cứng (argmax): ", values[np.argmax(keys @ query)])

Điểm càng "sắc" (nhân hệ số lớn), softmax càng gần **argmax** — attention tiến về tra cứu cứng. Điểm nhỏ → trả về gần trung bình mọi value. Độ lớn của điểm vì thế quyết định hành vi — điều này dẫn thẳng tới hệ số $\frac{1}{\sqrt{d_k}}$ ở mục 3.

## 2. Scaled dot-product attention

Với nhiều query cùng lúc (mỗi hàng của $Q$ là một query):

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V, \qquad Q \in \mathbb R^{n_q\times d_k},\; K \in \mathbb R^{n_k\times d_k},\; V \in \mathbb R^{n_k\times d_v}$$

Ma trận trọng số $A = \text{softmax}(\cdot)$ có kích thước $n_q\times n_k$, mỗi hàng tổng bằng 1. **Mask** (mục 4) đặt điểm của các cặp bị cấm thành $-\infty$ trước softmax → trọng số 0.

In [ ]:
def attention(Q, K, V, mask=None):
    """Scaled dot-product attention. mask: mảng bool (n_q, n_k), True = được phép nhìn."""
    d_k = Q.shape[-1]
    scores = Q @ K.swapaxes(-1, -2) / np.sqrt(d_k)                     # (..., n_q, n_k)
    if mask is not None:
        scores = np.where(mask, scores, -np.inf)
    A = softmax(scores, axis=-1)
    return A @ V, A


rng = np.random.default_rng(0)
Q, K, V = rng.normal(size=(3, 8)), rng.normal(size=(5, 8)), rng.normal(size=(5, 4))
out, A = attention(Q, K, V)
print("đầu ra:", out.shape, "| ma trận attention:", A.shape, "| tổng mỗi hàng:", A.sum(1).round(6))

ref = nn.functional.scaled_dot_product_attention(*(torch.tensor(m)[None] for m in (Q, K, V)))[0].numpy()
print(f"sai lệch so với torch.nn.functional.scaled_dot_product_attention: {np.abs(out - ref).max():.1e}")

## 3. Vì sao chia $\sqrt{d_k}$?

Nếu các thành phần của $\mathbf q, \mathbf k$ độc lập, trung bình 0, phương sai 1 thì $\mathbf q^\top\mathbf k = \sum_{i=1}^{d_k} q_ik_i$ có phương sai $d_k$ — độ lệch chuẩn $\sqrt{d_k}$. Với $d_k$ lớn, điểm rất lớn → softmax **bão hoà** (gần one-hot) → gradient qua softmax gần 0 (Jacobian của softmax là $\text{diag}(\mathbf p) - \mathbf p\mathbf p^\top$, tiến về 0 khi $\mathbf p$ gần one-hot). Chia $\sqrt{d_k}$ đưa phương sai về 1 với mọi $d_k$.

Thí nghiệm: query và 32 key ngẫu nhiên $\mathcal N(0, 1)$, đo độ lệch chuẩn của điểm, trọng số lớn nhất, entropy của phân phối attention (tối đa $\ln 32 \approx 3.47$) và độ lớn Jacobian của softmax:

In [ ]:
n_keys, rows = 32, []
for d in [4, 16, 64, 256, 1024]:
    q = rng.normal(size=(2000, d))
    Kd = rng.normal(size=(2000, n_keys, d))
    s = np.einsum("bd,bnd->bn", q, Kd)
    row = {"d_k": d, "std điểm": s.std()}
    for name, sc in [("không chia", s), ("chia √d", s / np.sqrt(d))]:
        p = softmax(sc)
        row[f"max trọng số ({name})"] = p.max(1).mean()
        row[f"entropy ({name})"] = -(p * np.log(p + 1e-30)).sum(1).mean()
        row[f"‖Jacobian‖ ({name})"] = np.mean([np.linalg.norm(np.diag(pi) - np.outer(pi, pi)) for pi in p[:300]])
    rows.append(row)

for r in rows:
    print(f"d_k={r['d_k']:5d} | std điểm {r['std điểm']:6.2f} | max trọng số {r['max trọng số (không chia)']:.3f} → {r['max trọng số (chia √d)']:.3f}"
          f" | entropy {r['entropy (không chia)']:.2f} → {r['entropy (chia √d)']:.2f}"
          f" | ‖Jacobian‖ {r['‖Jacobian‖ (không chia)']:.3f} → {r['‖Jacobian‖ (chia √d)']:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
ds = [r["d_k"] for r in rows]
for name, style in [("không chia", "o-"), ("chia √d", "s--")]:
    axes[0].plot(ds, [r[f"entropy ({name})"] for r in rows], style, label=name)
    axes[1].plot(ds, [r[f"‖Jacobian‖ ({name})"] for r in rows], style, label=name)
axes[0].axhline(np.log(n_keys), color="gray", lw=0.8, ls=":", label="ln 32 (đều)")
axes[0].set_ylabel("entropy của attention")
axes[1].set_ylabel("‖Jacobian softmax‖ (Frobenius)")
for ax in axes:
    ax.set_xscale("log", base=2)
    ax.set_xlabel("d_k")
    ax.legend()
plt.tight_layout()
plt.show()

Không chia: $d_k$ càng lớn, attention càng gần one-hot (entropy từ 2.27 xuống 0.11) và gradient qua softmax nhỏ dần (‖Jacobian‖ từ khoảng 0.29 ở $d_k \le 16$ xuống 0.06 ở $d_k = 1024$) — lúc khởi tạo, mô hình đã "chọn cứng" một cách ngẫu nhiên và học chậm. Chia $\sqrt{d_k}$: phân phối và gradient giữ nguyên tính chất với mọi $d_k$. Mô hình vẫn **học** được attention sắc nét nếu cần (bằng cách làm $W_Q, W_K$ lớn) — chỉ là không bị ép như vậy từ đầu.

## 4. Mask

- **Padding mask**: chuỗi trong batch có độ dài khác nhau được đệm; không token nào được chú ý tới vị trí đệm (cột bị che).
- **Causal mask** (tam giác dưới): token ở vị trí $i$ chỉ nhìn các vị trí $\le i$ — bắt buộc cho mô hình sinh (GPT, bài 03), nếu không mô hình "nhìn trộm" token cần dự đoán.

Trong code thực tế, hai mask thường được kết hợp bằng phép AND.

In [ ]:
T, length = 6, 4                                                        # chuỗi dài 4 đệm thành 6
padding_mask = np.broadcast_to(np.arange(T) < length, (T, T))           # che cột 4, 5
causal_mask = np.tril(np.ones((T, T), dtype=bool))

X = rng.normal(size=(T, 8))
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, (name, m) in zip(axes, [("padding", padding_mask), ("causal", causal_mask), ("padding AND causal", padding_mask & causal_mask)]):
    _, A = attention(X, X, X, mask=m)
    ax.imshow(A, cmap="Blues", vmin=0, vmax=1)
    ax.set_title(name)
    ax.set_xlabel("vị trí được nhìn (key)")
axes[0].set_ylabel("vị trí đang tính (query)")
plt.tight_layout()
plt.show()

with np.errstate(invalid="ignore"):
    _, A_bad = attention(X[:1], X, X, mask=np.zeros((1, T), dtype=bool))
print("hàng bị che toàn bộ →", A_bad.round(2), "(softmax của toàn −∞ = NaN: luôn để ít nhất một vị trí hợp lệ)")

## 5. Self-attention và tính hoán vị-đẳng biến

**Self-attention**: query, key, value đều lấy từ **cùng** một chuỗi $X \in \mathbb R^{n\times d}$ qua ba phép chiếu học được:
$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V, \qquad \text{đầu ra} = \text{Attention}(Q, K, V)$$
Mỗi token cập nhật biểu diễn của mình bằng thông tin từ **mọi** token khác — một bước, không qua chuỗi trạng thái như RNN.

Tính chất quan trọng: nếu hoán vị các hàng của $X$ (bằng ma trận hoán vị $P$), đầu ra bị hoán vị **đúng như vậy**: $\text{SA}(PX) = P\,\text{SA}(X)$. Self-attention coi chuỗi như một **tập hợp**: "chó cắn người" và "người cắn chó" cho cùng các biểu diễn (chỉ đổi chỗ).

In [ ]:
d_model = 8
W_Q, W_K, W_V = (rng.normal(size=(d_model, d_model)) / np.sqrt(d_model) for _ in range(3))


def self_attention(X):
    return attention(X @ W_Q, X @ W_K, X @ W_V)[0]


X = rng.normal(size=(5, d_model))
perm = rng.permutation(5)
print("hoán vị:", perm)
print(f"max |SA(PX) − P·SA(X)| = {np.abs(self_attention(X[perm]) - self_attention(X)[perm]).max():.1e}")

→ Self-attention **không biết thứ tự**. Muốn mô hình phân biệt vị trí, phải đưa thông tin vị trí vào biểu diễn: **positional encoding** (bài 02). Ngược lại, với dữ liệu vốn là tập hợp (các điểm trong đám mây điểm, các vật thể trong cảnh), tính chất này lại là ưu điểm.

## 6. Attention học được: bài toán tra cứu khoá–giá trị

Mỗi mẫu gồm 8 cặp (khoá, giá trị) — khoá là 8 số khác nhau trong 0–15, giá trị ngẫu nhiên trong 0–15 — và một **truy vấn** là một trong các khoá; đích là giá trị tương ứng. (Đây là dạng "associative recall" — phép tra cứu mà mô hình ngôn ngữ dùng liên tục, vd nhớ tên nhân vật đã nhắc ở đầu văn bản.)

So sánh hai mô hình:
- **Attention** một đầu: query từ khoá truy vấn, key/value từ từng cặp.
- **Gộp trung bình** (DeepSets): mỗi cặp qua một MLP, lấy trung bình, ghép với truy vấn rồi qua MLP — cũng nhìn được mọi cặp, nhưng phải "nén" chúng thành một vector trước khi biết truy vấn.

In [ ]:
N_PAIRS, N_KEYS, N_VALUES = 8, 16, 16


def recall_batch(n, g):
    keys = torch.stack([torch.randperm(N_KEYS, generator=g)[:N_PAIRS] for _ in range(n)])
    vals = torch.randint(0, N_VALUES, (n, N_PAIRS), generator=g)
    which = torch.randint(0, N_PAIRS, (n,), generator=g)
    rows = torch.arange(n)
    return keys, vals, keys[rows, which], vals[rows, which]


class RecallAttention(nn.Module):
    def __init__(self, d=32):
        super().__init__()
        self.emb_key, self.emb_val, self.emb_query = nn.Embedding(N_KEYS, d), nn.Embedding(N_VALUES, d), nn.Embedding(N_KEYS, d)
        self.W_Q, self.W_K, self.W_V = nn.Linear(d, d), nn.Linear(2 * d, d), nn.Linear(2 * d, d)
        self.out = nn.Linear(d, N_VALUES)

    def forward(self, keys, vals, query, return_weights=False):
        pairs = torch.cat([self.emb_key(keys), self.emb_val(vals)], dim=-1)    # (B, 8, 2d)
        q = self.W_Q(self.emb_query(query))[:, None]                           # (B, 1, d)
        k, v = self.W_K(pairs), self.W_V(pairs)                                # (B, 8, d)
        w = torch.softmax(q @ k.transpose(1, 2) / k.shape[-1] ** 0.5, dim=-1)  # (B, 1, 8)
        logits = self.out((w @ v)[:, 0])
        return (logits, w[:, 0]) if return_weights else logits


class RecallMeanPool(nn.Module):
    def __init__(self, d=32, hidden=256):
        super().__init__()
        self.emb_key, self.emb_val, self.emb_query = nn.Embedding(N_KEYS, d), nn.Embedding(N_VALUES, d), nn.Embedding(N_KEYS, d)
        self.phi = nn.Sequential(nn.Linear(2 * d, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.rho = nn.Sequential(nn.Linear(hidden + d, hidden), nn.ReLU(), nn.Linear(hidden, N_VALUES))

    def forward(self, keys, vals, query):
        pooled = self.phi(torch.cat([self.emb_key(keys), self.emb_val(vals)], dim=-1)).mean(1)
        return self.rho(torch.cat([pooled, self.emb_query(query)], dim=-1))


def train_recall(make, steps=3000, seed=0):
    torch.manual_seed(seed)
    model = make().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    g = torch.Generator().manual_seed(seed)
    for _ in range(steps):
        k, v, q, y = (t.to(device) for t in recall_batch(128, g))
        opt.zero_grad()
        nn.functional.cross_entropy(model(k, v, q), y).backward()
        opt.step()
    k, v, q, y = (t.to(device) for t in recall_batch(5000, torch.Generator().manual_seed(99)))
    with torch.no_grad():
        return model, (model(k, v, q).argmax(1) == y).float().mean().item()


recall_models = {}
for name, make in [("Attention", RecallAttention), ("Gộp trung bình", RecallMeanPool)]:
    t0 = time.perf_counter()
    recall_models[name], acc = train_recall(make)
    n_params = sum(p.numel() for p in recall_models[name].parameters())
    print(f"{name:15s}: {n_params:7,d} tham số | accuracy test = {acc:.3f} | {time.perf_counter() - t0:.0f}s")

In [ ]:
k, v, q, y = recall_batch(4, torch.Generator().manual_seed(7))
with torch.no_grad():
    _, w = recall_models["Attention"](k.to(device), v.to(device), q.to(device), return_weights=True)
fig, axes = plt.subplots(1, 4, figsize=(13, 2.6), sharey=True)
for i, ax in enumerate(axes):
    ax.bar(range(N_PAIRS), w[i].cpu(), color=["C3" if kk == q[i] else "C0" for kk in k[i]])
    ax.set_xticks(range(N_PAIRS), [f"{a}:{b}" for a, b in zip(k[i].tolist(), v[i].tolist())], rotation=60, fontsize=8)
    ax.set_title(f"truy vấn khoá {q[i].item()} → đáp án {y[i].item()}", fontsize=9)
axes[0].set_ylabel("trọng số attention")
plt.tight_layout()
plt.show()

Attention đạt gần 100% với ~7 nghìn tham số; mô hình gộp trung bình có hơn 160 nghìn tham số nhưng chỉ khoảng 45% (cao hơn mức đoán ngẫu nhiên 1/16 nhưng xa mức giải được). Biểu đồ: trọng số dồn gần hết vào **đúng cặp có khoá trùng truy vấn** (màu đỏ) — mô hình tự học phép tra cứu.

Bài học: khi thông tin cần lấy **phụ thuộc vào truy vấn**, nén mọi thứ thành một vector trước (gộp trung bình, trạng thái cuối của RNN) là nút cổ chai; attention cho phép **chọn lọc theo nội dung** tại thời điểm cần.

## 7. Chi phí: $O(n^2)$

Ma trận attention có $n\times n$ phần tử cho mỗi đầu, mỗi lớp. Bộ nhớ chỉ cho ma trận này (float32, 1 mẫu):

In [ ]:
for n in [512, 2048, 8192, 32768]:
    mb = n * n * 4 / 2**20
    print(f"n = {n:6,d} token: {mb:10,.0f} MB / đầu / lớp  (×12 đầu × 12 lớp = {mb * 144 / 1024:8,.1f} GB nếu giữ hết)")

Tính toán $O(n^2 d)$ và bộ nhớ $O(n^2)$ là giới hạn chính về độ dài ngữ cảnh. Hướng giải quyết: **FlashAttention** (tính theo khối, không lưu cả ma trận — cùng kết quả, ít bộ nhớ hơn; `scaled_dot_product_attention` của PyTorch tự chọn khi có thể), attention thưa/cửa sổ trượt, attention tuyến tính, các mô hình không gian trạng thái (SSM).

## 🧠 Kiểm tra nhanh

1. Viết công thức scaled dot-product attention. Kích thước của ma trận attention?
2. Vì sao chia $\sqrt{d_k}$? Điều gì xảy ra với softmax và gradient nếu không chia?
3. Padding mask và causal mask khác nhau thế nào? Khi nào cần mỗi loại?
4. "Self-attention hoán vị-đẳng biến" nghĩa là gì? Hệ quả?
5. Vì sao attention giải được bài toán tra cứu khoá–giá trị còn mô hình gộp trung bình thì khó?

## 📌 Tóm tắt

- Attention = tra cứu mềm: điểm $\mathbf q^\top\mathbf k$ → softmax → trung bình có trọng số của value; khả vi nên học được.
- Chia $\sqrt{d_k}$ giữ phương sai điểm ≈ 1, tránh softmax bão hoà và gradient nhỏ khi $d_k$ lớn.
- Mask đặt điểm $-\infty$: padding (bỏ vị trí đệm), causal (không nhìn tương lai); không để hàng bị che toàn bộ.
- Self-attention không biết thứ tự → cần mã hoá vị trí; chi phí $O(n^2)$ theo độ dài.

## 📚 Tài liệu tham khảo

- Vaswani et al. (2017) — *Attention Is All You Need*
- Bahdanau, Cho & Bengio (2015) — *Neural Machine Translation by Jointly Learning to Align and Translate*
- Zaheer et al. (2017) — *Deep Sets*
- Dao et al. (2022) — *FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness*
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) — Jay Alammar